<div style="background:linear-gradient(135deg,#0D1B2A 0%,#1565C0 100%);padding:36px 40px;border-radius:12px;margin-bottom:8px;">
  <div style="display:flex;align-items:center;gap:16px;">
    <span style="font-size:48px;">🛰️</span>
    <div>
      <p style="color:#ADE8F4;margin:0;font-size:13px;letter-spacing:3px;text-transform:uppercase;font-family:monospace;">Computer Vision for Remote Sensing · Bangladesh</p>
      <h1 style="color:#FFFFFF;margin:4px 0 0 0;font-size:28px;font-family:monospace;">Phase 1 · Raster CV Foundations</h1>
    </div>
  </div>
  <hr style="border-color:#1E88E5;margin:20px 0 16px 0;"/>
  <div style="display:flex;gap:32px;flex-wrap:wrap;">
    <span style="color:#90CAF9;font-family:monospace;font-size:13px;">📘 Lesson <b style='color:#FFF'>L1 / 7</b></span>
    <span style="color:#90CAF9;font-family:monospace;font-size:13px;">⏱ Est. Time <b style='color:#FFF'>2–3 hrs</b></span>
    <span style="color:#90CAF9;font-family:monospace;font-size:13px;">🔴 Priority <b style='color:#FFF'>Critical</b></span>
    <span style="color:#90CAF9;font-family:monospace;font-size:13px;">📍 Study Area <b style='color:#FFF'>Sitakunda, Bangladesh</b></span>
    <span style="color:#90CAF9;font-family:monospace;font-size:13px;">📅 Date <b style='color:#FFF'>____-__-__</b></span>
  </div>
</div>
<div style="background:#E3F2FD;border-left:5px solid #1565C0;padding:18px 24px;border-radius:0 8px 8px 0;">
  <h2 style="color:#1565C0;margin:0 0 8px 0;font-size:18px;">📖 L1 — Image as a Tensor</h2>
  <p style="margin:0;color:#1A2A3A;font-size:14px;line-height:1.7;">Satellite images are <b>multi-band tensors</b> with shape <code>(Bands, H, W)</code> — not the <code>(H, W, 3)</code> you know from OpenCV/PIL. Each band captures a different wavelength. DN (Digital Number) values are raw sensor counts, not visual colours. Always scale before computing indices.</p>
</div>

## 🎯 Objectives
By the end of this lesson you will be able to:
- ✅ Open a satellite GeoTIFF with `rasterio` and read it as a NumPy array
- ✅ Understand the `(Bands, H, W)` shape convention used in remote sensing
- ✅ Inspect CRS, pixel size, transform, dtype, and nodata value
- ✅ Re-order axes with `np.moveaxis()` for display and PyTorch compatibility

---
## 🔑 Key Functions
| Function | Purpose |
|---|---|
| `rasterio.open(path)` | Open a GeoTIFF |
| `src.read()` | Read all bands → `(Bands, H, W)` |
| `src.read(n)` | Read single band n (1-indexed) |
| `src.count` | Number of bands |
| `src.crs` | Coordinate Reference System |
| `src.transform` | Affine georeferencing transform |
| `src.res` | Pixel size `(x_res, y_res)` in map units |
| `src.nodata` | NoData fill value |
| `np.moveaxis(img, 0, -1)` | `(C,H,W)` → `(H,W,C)` for display |

---
## ⚙️ Setup & Imports

In [ ]:
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio import plot as rplot

# ── Paths ─────────────────────────────────────────────────────────────────
TIF_PATH = pathlib.Path("../../data/raw/sitakunda_sentinel2.tif")  # ← change
FIG_DIR  = pathlib.Path("../../outputs/figures/phase1")
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"rasterio : {rasterio.__version__}")
print(f"numpy    : {np.__version__}")
print(f"File OK  : {TIF_PATH.exists()}")

---
## 1 · Open a Raster and Read Its Shape

In [ ]:
with rasterio.open(TIF_PATH) as src:
    img = src.read()    # (Bands, H, W)

    print("=" * 52)
    print("  RASTER SUMMARY — Sitakunda Sentinel-2")
    print("=" * 52)
    print(f"  Shape  (Bands, H, W)  : {img.shape}")
    print(f"  Number of bands       : {src.count}")
    print(f"  Height × Width        : {src.height} × {src.width}")
    print(f"  Data type             : {src.dtypes[0]}")
    print(f"  CRS                   : {src.crs}")
    print(f"  EPSG code             : {src.crs.to_epsg()}")
    print(f"  Pixel size (x, y)     : {src.res[0]:.2f} m × {src.res[1]:.2f} m")
    print(f"  Bounding box          : {src.bounds}")
    print(f"  NoData value          : {src.nodata}")
    print(f"  DN range [Band 1]     : {img[0].min()} – {img[0].max()}")

---
## 2 · Understanding Band Dimensions

> **Key concept:** `rasterio` gives `(Bands, H, W)` — band-first.  
> `matplotlib` expects `(H, W, C)` — band-last.  
> PyTorch CNN expects `(N, C, H, W)` — batch-first, band-first.

Use `np.moveaxis()` to convert between them.

In [ ]:
print(f"From rasterio          (C, H, W) : {img.shape}")

img_hwc   = np.moveaxis(img, 0, -1)         # → (H, W, C)   for matplotlib
img_batch = img[np.newaxis, ...]             # → (1, C, H, W) for PyTorch

print(f"After moveaxis         (H, W, C) : {img_hwc.shape}")
print(f"With batch dim      (N, C, H, W) : {img_batch.shape}")

# Access individual bands (0-indexed)
blue = img[1]   # S2 Band 2 — Blue  490nm
red  = img[3]   # S2 Band 4 — Red   665nm
nir  = img[7]   # S2 Band 8 — NIR   842nm
print(f"\nSingle band shape: {blue.shape}  (H, W)")

---
## 3 · Visualize Individual Bands

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Sitakunda Sentinel-2 — Individual Bands', fontsize=14, fontweight='bold')

band_info = [
    (img[1], 'Band 2 — Blue  (490 nm)', 'Blues'),
    (img[3], 'Band 4 — Red   (665 nm)', 'Reds'),
    (img[7], 'Band 8 — NIR   (842 nm)', 'YlGn'),
]

for ax, (band, title, cmap) in zip(axes, band_info):
    lo, hi = np.percentile(band, 2), np.percentile(band, 98)
    ax.imshow(band, cmap=cmap, vmin=lo, vmax=hi)
    ax.set_title(title, fontsize=11)
    ax.axis('off')

plt.tight_layout()
plt.savefig(FIG_DIR / 'L1_bands.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved → outputs/figures/phase1/L1_bands.png")

---
## 🏋️ Exercise

<div style="background:#FFF9C4;border-left:4px solid #F9A825;padding:14px 20px;border-radius:0 8px 8px 0;">
<b>Task:</b> Open your Sitakunda <b>Landsat</b> TIF. Print its shape, dtype, CRS, and pixel size.<br/>
Questions to answer:
<ul style='margin:8px 0 0 0;'>
<li>What EPSG code is it in?</li>
<li>What is the pixel size compared to Sentinel-2?</li>
<li>How many bands does it have?</li>
</ul>
</div>

In [ ]:
# ── YOUR CODE HERE ────────────────────────────────────────────────────────
LANDSAT_PATH = pathlib.Path("../../data/raw/YOUR_LANDSAT_FILE.tif")   # ← change

# with rasterio.open(LANDSAT_PATH) as src:
#     img_l = src.read()
#     print(img_l.shape)
#     print(src.crs)
#     ...


---
## 📝 My Notes

> _Fill in after running the code._

| Property | Sentinel-2 | Landsat |
|---|---|---|
| CRS / EPSG | `____` | `____` |
| Pixel size | `____` m | `____` m |
| Num bands | `____` | `____` |
| DN range | `____` | `____` |

**Observations:** _(write here)_

---
<div style="background:linear-gradient(135deg,#1B5E20 0%,#2E7D32 100%);padding:24px 32px;border-radius:10px;">
  <h2 style="color:#FFFFFF;margin:0 0 12px 0;font-size:18px;">✅ Key Takeaways — L1</h2>
  <ul style="color:#C8E6C9;font-size:14px;line-height:2.2;margin:0;">
    <li>Rasterio reads as <code style='background:#1B5E20;padding:2px 6px;border-radius:4px;'>(Bands, H, W)</code> — always check shape before any operation</li>
    <li>DN values are raw counts — divide by 10,000 for Sentinel-2 L2A reflectance</li>
    <li>CRS + Transform = georeferencing — always preserve when writing outputs</li>
    <li><code style='background:#1B5E20;padding:2px 6px;border-radius:4px;'>np.moveaxis(img, 0, -1)</code> → <code>(H,W,C)</code> for matplotlib</li>
    <li>Bands are 1-indexed in rasterio, 0-indexed in NumPy arrays</li>
  </ul>
</div>

---
<div style="display:flex;justify-content:space-between;padding:10px 0;font-size:13px;color:#9E9E9E;">
  <span>← Previous: (none)</span>
  <b style="color:#1565C0;">Phase 1 · L1 of 7</b>
  <span><a href='L2_spatial_profile.ipynb' style='color:#1565C0;'>Next: L2 — Spatial Profile →</a></span>
</div>